# P06 · tendon vibration × polarity — the detailed analysis

Everything this session can be asked, on **every muscle recorded except the deltoid**, in one
place: how big the response is, and **when it arrives**.

**The design.** 8 recordings: 2 blocks (`VIB off` → `VIB ON`) × 2 polarities × 2 protocols.
Electrode 2, vibrator on the **left wrist extensor**. `VIB off` is the control — the vibrator
strapped on and silent — so it shares the setup, the electrodes and the elapsed time with
`VIB ON` and differs only in whether the motor runs.

**The rule for intensity.** Each panel (one protocol, one polarity) takes the threshold picked in
**its own `VIB off`** recording, and both of its blocks are read at it, muscle by muscle. Nothing
crosses between protocols or polarities.

**What is here**

| | |
|---|---|
| **§4** | **amplitude** — the response size, as in `P06_tendon_vibration_polarity.ipynb` but across every muscle |
| **§5** | **latency** — when the EMG leaves its baseline and when it peaks, with worked examples showing exactly how each number is taken |

**Why latency matters separately.** Size can change because more or fewer fibres were recruited.
A response that arrives **later** has taken a longer route — an extra synapse, or slower fibres.
It is the one measure here that speaks to the pathway rather than the gain.

**Sampling is ~0.8 ms**, so a latency difference under about **1.5 ms** is not resolvable. Do not
read a 1 ms shift.

> The log writes polarity as **positive / negative**; **negative is cathodic**.

**Colours:** `VIB off` **grey**, `VIB ON` **orange**. Muscles are drawn **left arm, then right**,
and the figures are split by side so thirteen muscles stay legible.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from functions import (set_style, load_run, pretty, thresholds_for, threshold_source,
                       latency_table, fig_vibration, fig_group_summary,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       mt_stamp, warn_if_stale, fig_threshold_grid, fig_detection_grid,
                       condition_table, snr_verdict, fig_noise_check,
                       fig_latency_example, fig_bars)
set_style()
from functions.paper import motor_thresholds

## 1 · Config

In [ ]:
# ======== the recordings ========
D = "tSCS_CHUV_data/29-09-2026/"    # P06, 29-09-2026. Only the 20260929 files are this session.

ALL = {   # (protocol, polarity, block) -> csv.  "negative" in the log = cathodic.
 ("30 Hz",  "cathodic", "VIB off"): D + "Burst_autosave_20260929_163809_190ms.csv",       # 16:38
 ("30 Hz",  "cathodic", "VIB ON"):  D + "Burst_autosave_20260929_175220_421ms.csv",       # 17:52
 ("30 Hz",  "anodic",   "VIB off"): D + "Burst_autosave_20260929_163517_916ms.csv",       # 16:35
 ("30 Hz",  "anodic",   "VIB ON"):  D + "Burst_autosave_20260929_170919_038ms.csv",       # 17:09
 ("ARC-EX", "cathodic", "VIB off"): D + "Modulated_autosave_20260929_164440_099ms.csv",   # 16:44
 ("ARC-EX", "cathodic", "VIB ON"):  D + "Modulated_autosave_20260929_170027_460ms.csv",   # 17:00
 ("ARC-EX", "anodic",   "VIB off"): D + "Modulated_autosave_20260929_164216_867ms.csv",   # 16:42
 ("ARC-EX", "anodic",   "VIB ON"):  D + "Modulated_autosave_20260929_173301_845ms.csv",   # 17:33
}

PROTOCOLS  = ["30 Hz", "ARC-EX"]
POLARITIES = ["cathodic", "anodic"]
CONTROL    = "VIB off"
CONDITIONS = [CONTROL, "VIB ON"]
PANELS     = [f"{p} · {pol}" for p in PROTOCOLS for pol in POLARITIES]
RUN        = {(f"{p} · {pol}", b): ALL[(p, pol, b)]
              for p in PROTOCOLS for pol in POLARITIES for b in CONDITIONS}

VIBRATED = "Ext. digitorum (L)"      # the vibrator sat on the left wrist extensor

# ======== every muscle in the recording except the deltoid ========
EXCLUDE = ("Deltoid",)          # analysed nowhere, and the one channel left out
MUSCLES = [pretty(c) for c in load_run(RUN[(PANELS[0], CONTROL)])[2]
           if c != "Trigger A" and not any(x.lower() in pretty(c).lower() for x in EXCLUDE)]
# read from the file rather than typed, so it cannot drift from what was recorded
PICK_MUSCLES, PICK_EXCLUDE = None, EXCLUDE      # the pickers offer exactly the same set

# ======== the latency settings ========
K_SD     = 3.0    # how far from its own baseline the EMG must go to count as having started
MIN_RUN  = 2      # ...and for how many samples in a row
JIT_MAX  = 3.0    # a muscle whose peak latency jitters more than this (ms) is not a latency
# Sampling is ~0.8 ms, so anything under ~1.5 ms is not resolvable.
# ======================================

COND_COLOUR = {CONTROL: "#8C8C8C", "VIB ON": "#E67E22"}

# thirteen muscles in one figure is unreadable, so every figure is drawn once per side
SIDES = {"left arm": [m for m in MUSCLES if m.endswith("(L)")],
         "right arm": [m for m in MUSCLES if m.endswith("(R)")]}

KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)

runs = RUN
for pan in PANELS:
    for b in CONDITIONS:
        f = runs[(pan, b)]
        amps = [m["amp_ma"] for m in load_run(f)[0]]
        print(f"{pan:20s} {b:10s} {f.split('/')[-1][-22:]}  {amps[0]:g}-{amps[-1]:g} mA"
              + ("   <- control" if b == CONTROL else ""))

## 2 · The motor thresholds you chose — every muscle but the deltoid

Each panel takes the threshold picked in its **`VIB off`** recording and reads both of its blocks
at it, muscle by muscle. Latency is measured at one intensity per muscle for the same reason
amplitude is: recruitment changes with current, and a response recruited from more fibres can peak
earlier even with no change in pathway.

`not picked` is expected at first — this notebook uses **every channel in the recording except the
deltoid**, where the others use four, so most of them need picking. Section 5 has one picker per
control recording and it covers the whole set at once.

The list is read from the file rather than typed, so it cannot drift from what was recorded.

In [ ]:
CHECK = {f"{pan} · {b}": runs[(pan, b)] for pan in PANELS for b in CONDITIONS}

picked = fig_threshold_grid(CHECK, MUSCLES, title="what is saved for each recording")

In [ ]:
MT = {pan: thresholds_for(runs[(pan, CONTROL)], MUSCLES, consecutive=2, **KW) for pan in PANELS}
for pan in PANELS:
    print(f"{pan:20s} {CONTROL:10s} {threshold_source(runs[(pan, CONTROL)])}")

USED = {f"{pan} · {b}": MT[pan] for pan in PANELS for b in CONDITIONS}
_ = fig_threshold_grid(CHECK, MUSCLES, thresholds=USED, share_gain=True,
                       title=f"the motor threshold used: each panel's {CONTROL}")

print(f"\n{'panel':20s}" + "".join(f"{m.split(' (')[0][:7] + m[-3:]:>13s}" for m in MUSCLES))
for pan in PANELS:
    print(f"{pan:20s}" + "".join(f"{str(MT[pan].get(m) or '-'):>13s}" for m in MUSCLES))

MT_STAMP = mt_stamp(runs)

## 3 · Where the latency is measured from

One panel per muscle at its own threshold, with the markers the latency is taken from. Check here
before reading section 4: a latency is only as good as the peak it is measured on, and a muscle
whose markers are scattered has no latency to report.

In [ ]:
TRY = dict(KW)
DET_MT = {f"{pan} · {b}": MT[pan] for pan in PANELS for b in CONDITIONS}
fig_detection_grid(CHECK, DET_MT, MUSCLES, edge_ms=1.0, jitter_ms=0.5,
                   title=f"peak detection at each panel's {CONTROL} threshold", **TRY)

## 4 · How big the response is

**Three bars per muscle.** Grey is `VIB off` at 100 %. Beside it, `VIB ON` measured two ways, both
as a % of `VIB off` measured the same way: its **1st pulse**, and the **mean of all 10 pulses**.
A manipulation can move one without the other.

`*` marks the vibrated muscle. Drawn once per arm, because thirteen muscles in one row cannot be
read.

**The noise under the vibrator is not the noise elsewhere** — it is a motor on the skin over
`Ext. digitorum (L)` and it raises that electrode's noise floor several fold. So `SNR_GATE` is off:
a reading under `SNR_SHOW` × its noise is marked `?` rather than removed.

In [ ]:
SNR_GATE, SNR_SHOW = None, 2.0

warn_if_stale(runs, MT_STAMP, used=[(pan, CONTROL) for pan in PANELS])
tab = condition_table(runs, MT, MUSCLES, CONDITIONS, base=CONTROL, match="common", **KW)

BARS = [CONTROL, "VIB ON · 1st pulse", "VIB ON · mean 1-10"]
BAR_COLOUR = {CONTROL: "#C6C6C6", "VIB ON · 1st pulse": "#F0A868",
              "VIB ON · mean 1-10": "#E67E22"}
both = {}
for pan in PANELS:
    for m in MUSCLES:
        if (pan, CONTROL, m) in tab:
            both[(pan, CONTROL, m)] = dict(tab[(pan, CONTROL, m)], pct=100.0)
        v = tab.get((pan, "VIB ON", m))
        if v:
            both[(pan, "VIB ON · 1st pulse", m)] = dict(v, pct=v.get("pct"))
            both[(pan, "VIB ON · mean 1-10", m)] = dict(v, pct=v.get("pct_all"))

for side, mus in SIDES.items():
    if not mus:
        continue
    fig_vibration(both, PANELS, mus, BARS, colours=BAR_COLOUR,
                  min_snr_ratio=SNR_GATE, zero_if_absent=True,
                  vibrated={pan: VIBRATED for pan in PANELS},
                  ylabel=f"% of {CONTROL}", ylim=(0, 200),
                  titles={pan: pan for pan in PANELS},
                  title=f"VIB ON as % of {CONTROL} — {side}", save=None)

### The numbers

In [ ]:
print(f"{'panel':20s}{'muscle':22s}{'mA':>5s}"
      + "".join(f"{b[:9] + ' 10p':>15s}{b[:9] + ' 1st':>15s}" for b in CONDITIONS))
for pan in PANELS:
    for m in MUSCLES:
        amp = next((tab[(pan, b, m)]["amp"] for b in CONDITIONS if (pan, b, m) in tab), None)
        if amp is None:
            continue
        cells = []
        for b in CONDITIONS:
            v, why = snr_verdict(tab, (pan, b, m), CONTROL, SNR_GATE)
            if why:
                cells += [why, ""]
            elif v is None:
                cells += ["-", "-"]
            else:
                mk = " ?" if v.get("snr", 9e9) < SNR_SHOW else "  "
                cells += [f"{v['pct_all']:.0f} %" + mk, f"{v['pct']:.0f} %" + mk]
        star = " *" if m == VIBRATED else "  "
        print(f"{pan:20s}{m + star:22s}{amp:5g}" + "".join(f"{c_:>15s}" for c_ in cells))
print(f"\n10p = mean of all 10 pulses   1st = the 1st pulse only, both as % of {CONTROL}")
print(f"?  = under {SNR_SHOW:g}x its own noise")

## 5 · When the response arrives

### 5a · How the two numbers are taken — worked examples

Before any latency is compared, here is exactly where it comes from. Each figure is one recording,
one muscle, the ten pulses re-aligned on their own onset:

| | |
|---|---|
| **grey band** | the baseline mean ± `K_SD` standard deviations of the pre-stimulus stretch |
| **thin green** | each pulse's own onset — the first moment it leaves that band and stays out for `MIN_RUN` samples |
| **thick green** | their mean, which is the **onset** reported |
| **v / ^** | the max and min the peak-to-peak is made of |
| **thick orange** | the mean of their midpoints, which is the **peak** reported |
| **red band** | the artifact, never searched |

**A wide grey band pushes the onset later** without anything changing in the muscle — which is
exactly what the vibrator does to the muscle it sits on. That is the measure's main weakness and
it is visible here rather than hidden.

In [ ]:
# ======== which examples to show ========
EXAMPLES = [("30 Hz · cathodic", CONTROL, "Flex. digitorum (R)"),   # a clean, strong response
            ("30 Hz · cathodic", "VIB ON", "Flex. digitorum (R)"),  # the same muscle, vibrator on
            ("30 Hz · cathodic", CONTROL, VIBRATED),                # the vibrated muscle, control
            ("30 Hz · cathodic", "VIB ON", VIBRATED)]               # ...and with the vibrator on
# ========================================

for pan, blk, m in EXAMPLES:
    amp = (lat if "lat" in dir() else {}).get((pan, blk, m), {}).get("amp") or MT[pan].get(m)
    if not amp:
        print(f"{pan} · {blk} · {m}: no threshold picked"); continue
    fig_latency_example(runs[(pan, blk)], m, amp, k_sd=K_SD, min_run=MIN_RUN,
                        title=f"{pan} · {blk} · {m}", **KW)

### 5b · Onset and peak, in milliseconds

**Not a percentage** — absolute times in ms after the pulse. Grey is `VIB off`, orange `VIB ON`.
A difference under **~1.5 ms** is below the sampling resolution and means nothing.

In [ ]:
lat = latency_table(runs, MT, MUSCLES, CONDITIONS, k_sd=K_SD, min_run=MIN_RUN,
                    match="common", **KW)

for side, mus in SIDES.items():
    if not mus:
        continue
    for key, lab in (("onset_ms", "onset — when the EMG leaves its baseline"),
                     ("peak_ms", "peak — the middle of the response")):
        fig_vibration(lat, PANELS, mus, CONDITIONS, key=key, colours=COND_COLOUR,
                      vibrated={pan: VIBRATED for pan in PANELS},
                      ylabel="ms after the pulse", ref=None, ylim=(0, 30),
                      titles={pan: pan for pan in PANELS},
                      title=f"{lab} — {side}", save=None)

### The numbers, with the jitter that says whether to trust them

In [ ]:
print(f"{'panel':20s}{'muscle':22s}{'mA':>5s}"
      + "".join(f"{b[:7] + ' onset':>15s}{b[:7] + ' peak':>14s}{b[:7] + ' jit':>13s}"
                for b in CONDITIONS))
for pan in PANELS:
    for m in MUSCLES:
        if not any((pan, b, m) in lat for b in CONDITIONS):
            continue
        row = ""
        for b in CONDITIONS:
            v = lat.get((pan, b, m), {})
            o, p_, j = v.get("onset_ms"), v.get("peak_ms"), v.get("jitter_ms")
            flag = " !" if (j is not None and j == j and j > JIT_MAX) else "  "
            row += (f"{(f'{o:.1f}' if o == o else '-'):>15s}"
                    f"{(f'{p_:.1f}' if p_ == p_ else '-'):>14s}"
                    f"{(f'{j:.1f}' if j == j else '-') + flag:>13s}")
        amp = next((lat[(pan, b, m)]["amp"] for b in CONDITIONS if (pan, b, m) in lat), None)
        star = " *" if m == VIBRATED else "  "
        print(f"{pan:20s}{m + star:22s}{amp:5g}{row}")
print("\nonset/peak in ms after the pulse; jit = SD of the peak latency over the 10 pulses")
print(f"!  = jitter over {JIT_MAX:g} ms, so that latency is not stable enough to compare")

### 5c · The change, in milliseconds

`VIB ON` minus `VIB off`, so a **positive bar means the response arrived later** with the vibrator
running. Zero is no change, and anything inside ±1.5 ms is the sampling resolution.

In [ ]:
delta = {}
for pan in PANELS:
    for m in MUSCLES:
        a, b = lat.get((pan, CONTROL, m)), lat.get((pan, "VIB ON", m))
        if not a or not b:
            continue
        delta[(pan, "VIB ON", m)] = dict(
            onset_ms=b.get("onset_ms", np.nan) - a.get("onset_ms", np.nan),
            peak_ms=b.get("peak_ms", np.nan) - a.get("peak_ms", np.nan),
            snr=min(a.get("snr", np.nan), b.get("snr", np.nan)))

for side, mus in SIDES.items():
    if not mus:
        continue
    for key, lab in (("onset_ms", "onset"), ("peak_ms", "peak")):
        fig_vibration(delta, PANELS, mus, ["VIB ON"], key=key, colours={"VIB ON": "#E67E22"},
                      vibrated={pan: VIBRATED for pan in PANELS},
                      ylabel="ms  (VIB ON − VIB off)", ref=0, ylim=(-8, 8),
                      titles={pan: pan for pan in PANELS},
                      title=f"change in {lab} latency — {side}", save=None)

### 5d · Pooled — does polarity behave differently?

The latency change averaged over the muscles, with every muscle as a dot coloured by its panel.
If vibration delays one polarity and not the other, they are not travelling the same route.

In [ ]:
DOT = {f"{p} · {pol}": c for (p, pol), c in
       zip([(p, pol) for p in PROTOCOLS for pol in POLARITIES],
           ["#4D4D4D", "#9A9A9A", "#8C2317", "#C0392B"])}

for key, lab in (("onset_ms", "onset"), ("peak_ms", "peak")):
    fig_group_summary(delta, PANELS, MUSCLES, ["VIB ON"], key=key,
                      colours={"VIB ON": "#B0B0B0"}, dot_colours=DOT, err="sd",
                      ylabel="ms  (VIB ON − VIB off)", ref=0, ylim=(-8, 8),
                      title=f"change in {lab} latency — every panel, muscle by muscle", save=None)

## 5 · Choose the motor thresholds

**Four cells**, one per panel — the `VIB off` recording, which sets the intensity for both of that
panel's blocks. Each picker shows **every muscle but the deltoid**, so one pass per cell covers them.

Move the slider, press **Set to slider**, then **Save**, and re-run from section 2.

In [ ]:
def repick(key):
    """Open the picker for one recording, with a Save button. All eight muscles at once."""
    csv = runs[key]
    meta_, t_, sig_ = load_run(csv)
    # every channel in the file except the ones in PICK_EXCLUDE, not only the muscles this
    # notebook analyses - a threshold saved costs nothing and is there if MUSCLES is widened.
    chans = [c for c in sig_ if c != "Trigger A"
             and (PICK_MUSCLES is None or pretty(c) in PICK_MUSCLES)
             and not any(x.lower() in pretty(c).lower() for x in PICK_EXCLUDE)]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
repick(('30 Hz · cathodic', 'VIB off'))

In [ ]:
repick(('30 Hz · anodic', 'VIB off'))

In [ ]:
repick(('ARC-EX · cathodic', 'VIB off'))

In [ ]:
repick(('ARC-EX · anodic', 'VIB off'))